# CO3133 - Text-Image Classification with N24News

Notebook n?y tri?n khai b?i to?n **multimodal news classification** tr?n `N24News`.
??u v?o c?a m?i m?u l? **?nh b?i b?o + v?n b?n b?i b?o**, ??u ra l? **nh?n chuy?n m?c tin t?c**.
Hai m? h?nh ???c so s?nh l? `CLIP` v? `VisualBERT`.


### Dependency Setup

Cell d??i ??y c?i c?c th? vi?n c?n thi?t ?? notebook c? th? ch?y tr?n m?t m?y m?i.


In [ ]:
# Run this cell only if your environment is missing the required packages.
%pip install -q pandas numpy matplotlib seaborn pillow scikit-learn     torch torchvision torchaudio transformers accelerate gdown tqdm


### Import Libraries And Resolve Paths

Cell n?y import c?c th? vi?n ch?nh v? d? ??ng th? m?c `btl1/` m? kh?ng ph? thu?c v?o `GPT.md` hay file ngo?i lu?ng.


In [ ]:
from __future__ import annotations

import json
import random
import re
import tarfile
import zipfile
from pathlib import Path
from typing import Iterable

import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoImageProcessor,
    AutoProcessor,
    AutoTokenizer,
    CLIPModel,
    CLIPVisionModel,
    VisualBertConfig,
    VisualBertModel,
    get_linear_schedule_with_warmup,
)


def find_btl1_root(start: Path) -> Path:
    start = start.resolve()
    for path in [start, *start.parents]:
        if path.name == "btl1" and (path / "notebooks").exists():
            return path
        candidate = path / "btl1"
        if candidate.exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not locate the btl1/ directory from the current working directory.")


BTL1_ROOT = find_btl1_root(Path.cwd())
REPO_ROOT = BTL1_ROOT.parent
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"BTL1 root: {BTL1_ROOT}")
print(f"Repo root: {REPO_ROOT}")
print(f"Device: {DEVICE}")


### Configure Dataset Paths And Experiment Settings

Cell n?y ??nh ngh?a n?i ??t `source/raw/processed/artifacts`, link Google Drive c?a `N24News`, c?c t? l? chia t?p d? li?u v? si?u tham s? cho `CLIP` v? `VisualBERT`.


In [ ]:
N24_DIR = BTL1_ROOT / "data" / "multimodal" / "n24news"
SOURCE_DIR = N24_DIR / "source"
RAW_DIR = N24_DIR / "raw"
PROCESSED_DIR = N24_DIR / "processed"
ARTIFACT_DIR = BTL1_ROOT / "artifacts" / "multimodal"

N24NEWS_DRIVE_URL = "https://drive.google.com/file/d/1OS1fXwZ1Vsj70lEQajccyssxQRYp5X9D/view?usp=share_link"
N24NEWS_DRIVE_FILE_ID = "1OS1fXwZ1Vsj70lEQajccyssxQRYp5X9D"
OFFICIAL_REPO_URL = "https://github.com/billywzh717/N24News"
PAPER_URL = "https://aclanthology.org/2022.lrec-1.729/"

SEED = 42
DEV_RATIO = 0.10
TEST_RATIO = 0.20
MIN_TEXT_CHARS = 40
TOP_K_CLASSES = None  # Set an integer to keep only the most frequent classes.
MAX_SAMPLES = None  # Set an integer for a smaller debug subset.
DOWNLOAD_IF_MISSING = True
REBUILD_PROCESSED = False
NUM_WORKERS = 0  # Keep Windows-friendly dataloaders.

CLIP_CFG = {
    "backbone": "openai/clip-vit-base-patch32",
    "batch_size": 8 if torch.cuda.is_available() else 4,
    "epochs": 2,
    "lr": 1e-5,
    "weight_decay": 0.01,
    "dropout": 0.2,
    "max_text_length": 77,
    "warmup_ratio": 0.1,
    "patience": 2,
}

VB_CFG = {
    "visualbert_backbone": "uclanlp/visualbert-vqa-coco-pre",
    "vision_backbone": "openai/clip-vit-base-patch32",
    "tokenizer_name": "bert-base-uncased",
    "batch_size": 4 if torch.cuda.is_available() else 2,
    "epochs": 2,
    "lr": 1e-5,
    "weight_decay": 0.01,
    "dropout": 0.2,
    "max_text_length": 256,
    "max_visual_tokens": 32,
    "warmup_ratio": 0.1,
    "patience": 2,
}

for folder in [SOURCE_DIR, RAW_DIR, PROCESSED_DIR, ARTIFACT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Source directory: {SOURCE_DIR}")
print(f"Raw directory: {RAW_DIR}")
print(f"Processed directory: {PROCESSED_DIR}")
print(f"Artifact directory: {ARTIFACT_DIR}")


### Helper Functions For Data Acquisition And Parsing

Cell n?y x? l? t?i d? li?u, gi?i n?n archive, t?m manifest JSON, l?p ch? m?c ?nh v? h?p nh?t v?n b?n b?i b?o.


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)


IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
ARCHIVE_SUFFIXES = (".zip", ".tar", ".tar.gz", ".tgz")
CATEGORY_KEYS = ("category", "label", "class", "section", "topic")
IMAGE_KEYS = ("image_id", "img_id", "image", "imageid", "img")
HEADLINE_KEYS = ("headline", "title")
ABSTRACT_KEYS = ("abstract", "summary", "snippet", "lead")
BODY_KEYS = ("body", "article", "maintext", "text", "content")
CAPTION_KEYS = ("caption", "image_caption")
ID_KEYS = ("article_id", "news_id", "id", "doc_id")


def normalize_space(text: str) -> str:
    text = text.replace("\n", " ").replace("\r", " ")
    return re.sub(r"\s+", " ", text).strip()


def pick_field(record: dict, keys: Iterable[str]) -> str:
    for key in keys:
        value = record.get(key)
        if isinstance(value, str) and normalize_space(value):
            return normalize_space(value)
        if isinstance(value, (int, float)):
            return str(value)
    return ""


def extract_records(payload) -> list[dict]:
    if isinstance(payload, list):
        return [item for item in payload if isinstance(item, dict)]
    if isinstance(payload, dict):
        for key in ["data", "items", "articles", "news", "records", "dataset"]:
            value = payload.get(key)
            if isinstance(value, list) and value and isinstance(value[0], dict):
                return value
        if payload and all(isinstance(v, dict) for v in payload.values()):
            return list(payload.values())
    return []


def record_score(record: dict) -> int:
    score = 0
    if pick_field(record, CATEGORY_KEYS):
        score += 3
    if pick_field(record, IMAGE_KEYS):
        score += 3
    if pick_field(record, HEADLINE_KEYS):
        score += 2
    if pick_field(record, BODY_KEYS) or pick_field(record, ABSTRACT_KEYS):
        score += 2
    return score


def download_from_drive_if_missing() -> None:
    archives = [path for path in SOURCE_DIR.iterdir() if path.is_file()]
    if archives:
        print("Source file already exists. Skipping Google Drive download.")
        return
    if not DOWNLOAD_IF_MISSING:
        raise FileNotFoundError(
            "N24News source file is missing. Place the downloaded archive or extracted files in "
            "btl1/data/multimodal/n24news/source/."
        )
    print("Downloading N24News from Google Drive...")
    gdown.download(url=N24NEWS_DRIVE_URL, output=str(SOURCE_DIR), fuzzy=True, quiet=False)


def find_archives(folder: Path) -> list[Path]:
    files = []
    for path in folder.iterdir():
        if not path.is_file():
            continue
        if path.name.lower().endswith(ARCHIVE_SUFFIXES):
            files.append(path)
    return sorted(files)


def extract_archive(archive_path: Path, target_dir: Path) -> None:
    marker_files = list(target_dir.rglob("*.json")) + list(target_dir.rglob("*.jpg"))
    if marker_files:
        print(f"Raw directory already contains extracted data. Skipping extraction for {archive_path.name}.")
        return

    print(f"Extracting {archive_path.name}...")
    if archive_path.name.lower().endswith(".zip"):
        with zipfile.ZipFile(archive_path, "r") as zf:
            zf.extractall(target_dir)
    elif archive_path.name.lower().endswith((".tar.gz", ".tgz", ".tar")):
        mode = "r:gz" if archive_path.name.lower().endswith((".tar.gz", ".tgz")) else "r"
        with tarfile.open(archive_path, mode) as tf:
            tf.extractall(target_dir)
    else:
        raise ValueError(f"Unsupported archive format: {archive_path}")
    print("Extraction complete.")


def ensure_raw_data_available() -> None:
    if list(RAW_DIR.rglob("*.json")) and list(RAW_DIR.rglob("*.jpg")):
        print("Raw N24News data already available.")
        return

    source_json = list(SOURCE_DIR.rglob("*.json"))
    source_images = []
    for ext in IMAGE_EXTENSIONS:
        source_images.extend(SOURCE_DIR.rglob(f"*{ext}"))
    if source_json and source_images:
        print("Detected extracted source data. Using files directly from source/.")
        return

    download_from_drive_if_missing()
    archives = find_archives(SOURCE_DIR)
    if not archives:
        raise FileNotFoundError(
            "No N24News archive was found in source/. Place the downloaded file there or enable auto-download."
        )
    for archive in archives:
        extract_archive(archive, RAW_DIR)


def candidate_data_roots() -> list[Path]:
    roots = []
    if RAW_DIR.exists():
        roots.append(RAW_DIR)
    if SOURCE_DIR.exists():
        roots.append(SOURCE_DIR)
    return roots


def find_manifest_path() -> Path:
    candidates = []
    for root in candidate_data_roots():
        for json_path in root.rglob("*.json"):
            try:
                payload = json.loads(json_path.read_text(encoding="utf-8"))
            except Exception:
                try:
                    payload = json.loads(json_path.read_text(encoding="latin-1"))
                except Exception:
                    continue
            records = extract_records(payload)
            if not records:
                continue
            scores = [record_score(item) for item in records[: min(len(records), 200)]]
            score = sum(scores) / max(len(scores), 1)
            candidates.append((score, len(records), json_path))
    if not candidates:
        raise FileNotFoundError(
            "Could not locate a usable N24News json manifest. Check the downloaded dataset layout."
        )
    candidates.sort(reverse=True)
    best = candidates[0][2]
    print(f"Using manifest: {best}")
    return best


def load_manifest_records(manifest_path: Path) -> list[dict]:
    try:
        payload = json.loads(manifest_path.read_text(encoding="utf-8"))
    except UnicodeDecodeError:
        payload = json.loads(manifest_path.read_text(encoding="latin-1"))
    records = extract_records(payload)
    if not records:
        raise RuntimeError(f"Manifest file {manifest_path} did not contain a usable list of article records.")
    return records


def build_image_index() -> dict[str, str]:
    image_files = []
    for root in candidate_data_roots():
        for ext in IMAGE_EXTENSIONS:
            image_files.extend(root.rglob(f"*{ext}"))
    if not image_files:
        raise FileNotFoundError("No image files were found for N24News.")

    index = {}
    for path in sorted(image_files):
        stem = path.stem
        rel_path = path.relative_to(REPO_ROOT).as_posix() if REPO_ROOT in path.parents else path.relative_to(BTL1_ROOT).as_posix()
        index.setdefault(stem, rel_path)
    print(f"Indexed {len(index):,} unique image stems.")
    return index


def compose_text(record: dict) -> tuple[str, dict[str, str]]:
    headline = pick_field(record, HEADLINE_KEYS)
    abstract = pick_field(record, ABSTRACT_KEYS)
    body = pick_field(record, BODY_KEYS)
    caption = pick_field(record, CAPTION_KEYS)

    blocks = []
    if headline:
        blocks.append(f"Headline: {headline}")
    if abstract:
        blocks.append(f"Abstract: {abstract}")
    if body:
        blocks.append(f"Body: {body}")
    if caption:
        blocks.append(f"Caption: {caption}")

    return "\n".join(blocks).strip(), {
        "headline": headline,
        "abstract": abstract,
        "body": body,
        "caption": caption,
    }


### Build The Processed Train Dev Test Splits

Cell n?y s? ki?m tra d? li?u raw, parse article record, gh?p `image + text`, l?c m?u l?i, r?i chia `train/dev/test` theo stratified split.


In [ ]:
base_columns = [
    "article_id",
    "image_id",
    "category",
    "label_id",
    "text_input",
    "headline",
    "abstract",
    "body",
    "caption",
    "image_relpath",
    "text_chars",
]

split_paths = {
    "train": PROCESSED_DIR / "train.csv",
    "dev": PROCESSED_DIR / "dev.csv",
    "test": PROCESSED_DIR / "test.csv",
}


def build_rows_from_records(records: list[dict], image_index: dict[str, str], split_name: str) -> list[dict]:
    rows = []
    for idx, record in enumerate(tqdm(records, desc=f"Parsing {split_name} records")):
        category = pick_field(record, CATEGORY_KEYS)
        image_id = pick_field(record, IMAGE_KEYS)
        article_id = pick_field(record, ID_KEYS) or f"{split_name}_{idx}"
        text_input, text_parts = compose_text(record)

        if not category or not image_id or len(text_input) < MIN_TEXT_CHARS:
            continue

        image_relpath = image_index.get(Path(str(image_id)).stem)
        if image_relpath is None:
            continue

        rows.append(
            {
                "article_id": article_id,
                "image_id": str(image_id),
                "category": category,
                "text_input": text_input,
                "headline": text_parts["headline"],
                "abstract": text_parts["abstract"],
                "body": text_parts["body"],
                "caption": text_parts["caption"],
                "image_relpath": image_relpath,
                "text_chars": len(text_input),
            }
        )
    return rows


if all(path.exists() for path in split_paths.values()) and not REBUILD_PROCESSED:
    train_df = pd.read_csv(split_paths["train"])
    dev_df = pd.read_csv(split_paths["dev"])
    test_df = pd.read_csv(split_paths["test"])
    label_mapping = (
        pd.concat([train_df[["label_id", "category"]], dev_df[["label_id", "category"]], test_df[["label_id", "category"]]])
        .drop_duplicates()
        .sort_values("label_id")
    )
    label_names = label_mapping["category"].tolist()
    print("Loaded existing processed splits.")
else:
    ensure_raw_data_available()
    image_index = build_image_index()

    official_split_files = {}
    for root in candidate_data_roots():
        for split_name in ["train", "dev", "test"]:
            split_file = next(root.rglob(f"nytimes_{split_name}.json"), None)
            if split_file is not None:
                official_split_files[split_name] = split_file

    if len(official_split_files) == 3:
        print("Using official N24News train/dev/test files.")
        train_records = load_manifest_records(official_split_files["train"])
        dev_records = load_manifest_records(official_split_files["dev"])
        test_records = load_manifest_records(official_split_files["test"])

        train_df = pd.DataFrame(build_rows_from_records(train_records, image_index, "train"))
        dev_df = pd.DataFrame(build_rows_from_records(dev_records, image_index, "dev"))
        test_df = pd.DataFrame(build_rows_from_records(test_records, image_index, "test"))
        combined_df = pd.concat([train_df, dev_df, test_df], ignore_index=True)
    else:
        manifest_path = find_manifest_path()
        records = load_manifest_records(manifest_path)
        combined_df = pd.DataFrame(build_rows_from_records(records, image_index, "full"))
        if combined_df.empty:
            raise RuntimeError("No valid multimodal records were parsed from N24News.")

        category_counts = combined_df["category"].value_counts()
        if TOP_K_CLASSES is not None:
            keep_categories = category_counts.head(TOP_K_CLASSES).index.tolist()
            combined_df = combined_df[combined_df["category"].isin(keep_categories)].reset_index(drop=True)
            print(f"Keeping top-{TOP_K_CLASSES} categories.")

        if MAX_SAMPLES is not None and len(combined_df) > MAX_SAMPLES:
            combined_df = (
                combined_df.groupby("category", group_keys=False)
                .apply(lambda frame: frame.sample(min(len(frame), max(1, MAX_SAMPLES // max(combined_df['category'].nunique(), 1))), random_state=SEED))
                .reset_index(drop=True)
            )
            print(f"Using debug subset with {len(combined_df):,} samples.")

        temp_ratio = DEV_RATIO + TEST_RATIO
        train_df, temp_df = train_test_split(
            combined_df,
            test_size=temp_ratio,
            random_state=SEED,
            stratify=combined_df["category"],
        )
        dev_df, test_df = train_test_split(
            temp_df,
            test_size=TEST_RATIO / temp_ratio,
            random_state=SEED,
            stratify=temp_df["category"],
        )

    if TOP_K_CLASSES is not None:
        keep_categories = combined_df["category"].value_counts().head(TOP_K_CLASSES).index.tolist()
        train_df = train_df[train_df["category"].isin(keep_categories)].reset_index(drop=True)
        dev_df = dev_df[dev_df["category"].isin(keep_categories)].reset_index(drop=True)
        test_df = test_df[test_df["category"].isin(keep_categories)].reset_index(drop=True)
        combined_df = pd.concat([train_df, dev_df, test_df], ignore_index=True)

    label_names = combined_df["category"].value_counts().index.tolist()
    label_to_id = {label: idx for idx, label in enumerate(label_names)}

    for frame in [train_df, dev_df, test_df]:
        frame["label_id"] = frame["category"].map(label_to_id)

    train_df = train_df[base_columns].reset_index(drop=True)
    dev_df = dev_df[base_columns].reset_index(drop=True)
    test_df = test_df[base_columns].reset_index(drop=True)

    train_df.to_csv(split_paths["train"], index=False)
    dev_df.to_csv(split_paths["dev"], index=False)
    test_df.to_csv(split_paths["test"], index=False)
    print("Saved processed splits to:", PROCESSED_DIR)

split_frames = {"train": train_df, "dev": dev_df, "test": test_df}
for split_name, frame in split_frames.items():
    print(f"{split_name:>5}: {len(frame):,} samples")
print("Categories:", label_names)


### Summarize The Prepared Dataset

Cell n?y t?o th?ng k? ph?n b? l?p v? ?? d?i v?n b?n ?? ph?c v? EDA v? b?o c?o.


In [ ]:
label_summary_rows = []
for split_name, frame in split_frames.items():
    counts = frame["category"].value_counts()
    for category, count in counts.items():
        label_summary_rows.append(
            {
                "split": split_name,
                "category": category,
                "samples": int(count),
            }
        )

label_summary_df = pd.DataFrame(label_summary_rows)
label_summary_path = ARTIFACT_DIR / "n24news_label_summary.csv"
label_summary_df.to_csv(label_summary_path, index=False)

length_summary_df = pd.concat(
    [frame.assign(split=split_name)[["split", "text_chars"]] for split_name, frame in split_frames.items()],
    ignore_index=True,
)
length_summary_path = ARTIFACT_DIR / "n24news_text_length_summary.csv"
length_summary_df.to_csv(length_summary_path, index=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

plot_counts = (
    label_summary_df.groupby("category", as_index=False)["samples"]
    .sum()
    .sort_values("samples", ascending=False)
)
sns.barplot(data=plot_counts, x="samples", y="category", ax=axes[0], palette="crest")
axes[0].set_title("Category distribution")
axes[0].set_xlabel("Samples")
axes[0].set_ylabel("Category")

sns.histplot(data=length_summary_df, x="text_chars", hue="split", bins=40, element="step", ax=axes[1])
axes[1].set_title("Text length distribution")
axes[1].set_xlabel("Characters in fused text")
axes[1].set_ylabel("Articles")

fig.tight_layout()
eda_path = ARTIFACT_DIR / "n24news_eda_overview.png"
fig.savefig(eda_path, dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"Saved label summary: {label_summary_path}")
print(f"Saved text summary: {length_summary_path}")
print(f"Saved EDA figure: {eda_path}")


### Create Dataset Objects And Batch Builders

Cell n?y chuy?n c?c CSV ?? x? l? th?nh `PyTorch Dataset` v? chu?n b? hai lu?ng collate ri?ng cho `CLIP` v? `VisualBERT`.


In [ ]:
class N24NewsDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        image_path = REPO_ROOT / row["image_relpath"]
        with Image.open(image_path) as image:
            image = image.convert("RGB")
        return {
            "article_id": row["article_id"],
            "image": image,
            "text": row["text_input"],
            "label_id": int(row["label_id"]),
        }


clip_processor = AutoProcessor.from_pretrained(CLIP_CFG["backbone"])
vision_processor = AutoImageProcessor.from_pretrained(VB_CFG["vision_backbone"])
tokenizer = AutoTokenizer.from_pretrained(VB_CFG["tokenizer_name"])

train_dataset = N24NewsDataset(train_df)
dev_dataset = N24NewsDataset(dev_df)
test_dataset = N24NewsDataset(test_df)


def clip_collate(batch: list[dict]) -> tuple[dict, torch.Tensor]:
    images = [item["image"] for item in batch]
    texts = [item["text"] for item in batch]
    labels = torch.tensor([item["label_id"] for item in batch], dtype=torch.long)
    encoded = clip_processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=CLIP_CFG["max_text_length"],
    )
    return encoded, labels


def visualbert_collate(batch: list[dict]) -> tuple[dict, torch.Tensor]:
    images = [item["image"] for item in batch]
    texts = [item["text"] for item in batch]
    labels = torch.tensor([item["label_id"] for item in batch], dtype=torch.long)

    text_inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=VB_CFG["max_text_length"],
    )
    pixel_values = vision_processor(images=images, return_tensors="pt")["pixel_values"]
    batch_inputs = {
        "input_ids": text_inputs["input_ids"],
        "attention_mask": text_inputs["attention_mask"],
        "token_type_ids": text_inputs.get("token_type_ids", torch.zeros_like(text_inputs["input_ids"])),
        "pixel_values": pixel_values,
    }
    return batch_inputs, labels


def make_loader(dataset: Dataset, batch_size: int, collate_fn, shuffle: bool) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        collate_fn=collate_fn,
    )


print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(dev_dataset))
print("Test dataset size:", len(test_dataset))


### Define The CLIP And VisualBERT Classifiers

Cell n?y b?c backbone pretrained b?ng classification head ?? c? hai m? h?nh c?ng gi?i b?i to?n news classification tr?n `N24News`.


In [ ]:
class CLIPClassifier(nn.Module):
    def __init__(self, backbone: str, num_labels: int, dropout: float):
        super().__init__()
        self.backbone = CLIPModel.from_pretrained(backbone)
        hidden_size = self.backbone.config.projection_dim
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size * 2, num_labels)

    def forward(self, input_ids, attention_mask, pixel_values):
        image_features = self.backbone.get_image_features(pixel_values=pixel_values)
        text_features = self.backbone.get_text_features(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        image_features = nn.functional.normalize(image_features, dim=-1)
        text_features = nn.functional.normalize(text_features, dim=-1)
        fused = torch.cat([image_features, text_features], dim=-1)
        return self.classifier(self.dropout(fused))


class VisualBERTClassifier(nn.Module):
    def __init__(self, visualbert_backbone: str, vision_backbone: str, num_labels: int, dropout: float, max_visual_tokens: int):
        super().__init__()
        self.vision_encoder = CLIPVisionModel.from_pretrained(vision_backbone)
        config = VisualBertConfig.from_pretrained(visualbert_backbone)
        config.visual_embedding_dim = self.vision_encoder.config.hidden_size
        self.backbone = VisualBertModel.from_pretrained(visualbert_backbone, config=config, ignore_mismatched_sizes=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(config.hidden_size, num_labels)
        self.max_visual_tokens = max_visual_tokens

    def forward(self, input_ids, attention_mask, token_type_ids, pixel_values):
        visual_outputs = self.vision_encoder(pixel_values=pixel_values, return_dict=True)
        visual_embeds = visual_outputs.last_hidden_state[:, 1 : 1 + self.max_visual_tokens, :]
        batch_size, visual_tokens, _ = visual_embeds.shape
        visual_attention_mask = torch.ones((batch_size, visual_tokens), dtype=torch.long, device=visual_embeds.device)
        visual_token_type_ids = torch.ones((batch_size, visual_tokens), dtype=torch.long, device=visual_embeds.device)

        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            visual_embeds=visual_embeds,
            visual_attention_mask=visual_attention_mask,
            visual_token_type_ids=visual_token_type_ids,
            return_dict=True,
        )
        pooled = outputs.pooler_output if outputs.pooler_output is not None else outputs.last_hidden_state[:, 0]
        return self.classifier(self.dropout(pooled))


### Add Training And Evaluation Utilities

Cell n?y gom logic hu?n luy?n, validation, early stopping, l?u checkpoint v? ??nh gi? cu?i tr?n t?p test.


In [ ]:
def move_batch_to_device(batch_inputs: dict, labels: torch.Tensor) -> tuple[dict, torch.Tensor]:
    batch_inputs = {key: value.to(DEVICE) for key, value in batch_inputs.items()}
    labels = labels.to(DEVICE)
    return batch_inputs, labels


def collect_predictions(model: nn.Module, loader: DataLoader, criterion: nn.Module) -> tuple[float, np.ndarray, np.ndarray]:
    model.eval()
    total_loss = 0.0
    total_examples = 0
    logits_list = []
    labels_list = []

    with torch.no_grad():
        for batch_inputs, labels in loader:
            batch_inputs, labels = move_batch_to_device(batch_inputs, labels)
            logits = model(**batch_inputs)
            loss = criterion(logits, labels)
            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_examples += batch_size
            logits_list.append(logits.detach().cpu().numpy())
            labels_list.append(labels.detach().cpu().numpy())

    mean_loss = total_loss / max(total_examples, 1)
    logits_np = np.concatenate(logits_list, axis=0)
    labels_np = np.concatenate(labels_list, axis=0)
    return mean_loss, logits_np, labels_np


def compute_metrics(logits: np.ndarray, labels: np.ndarray, label_names: list[str]) -> tuple[dict, pd.DataFrame, np.ndarray]:
    predictions = logits.argmax(axis=1)
    metrics = {
        "accuracy": float(accuracy_score(labels, predictions)),
        "macro_f1": float(f1_score(labels, predictions, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(labels, predictions, average="weighted", zero_division=0)),
    }
    per_label_f1 = f1_score(labels, predictions, average=None, labels=list(range(len(label_names))), zero_division=0)
    support = pd.Series(labels).value_counts().reindex(range(len(label_names)), fill_value=0).to_numpy()
    per_label_df = pd.DataFrame(
        {
            "category": label_names,
            "f1": per_label_f1,
            "support": support.astype(int),
        }
    )
    cm = confusion_matrix(labels, predictions, labels=list(range(len(label_names))))
    return metrics, per_label_df, cm


def plot_history(history_df: pd.DataFrame, model_name: str) -> Path:
    figure_path = ARTIFACT_DIR / f"n24news_{model_name.lower()}_history.png"
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train")
    axes[0].plot(history_df["epoch"], history_df["dev_loss"], marker="o", label="dev")
    axes[0].set_title(f"{model_name} loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()

    axes[1].plot(history_df["epoch"], history_df["dev_accuracy"], marker="o", label="dev accuracy")
    axes[1].plot(history_df["epoch"], history_df["dev_macro_f1"], marker="o", label="dev macro F1")
    axes[1].set_title(f"{model_name} validation metrics")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Score")
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(figure_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return figure_path


def plot_confusion(cm: np.ndarray, model_name: str, label_names: list[str]) -> Path:
    figure_path = ARTIFACT_DIR / f"n24news_{model_name.lower()}_confusion_matrix.png"
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(cm, cmap="Blues", ax=ax, square=False)
    ax.set_title(f"{model_name} confusion matrix")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks(np.arange(len(label_names)) + 0.5)
    ax.set_xticklabels(label_names, rotation=90)
    ax.set_yticks(np.arange(len(label_names)) + 0.5)
    ax.set_yticklabels(label_names, rotation=0)
    fig.tight_layout()
    fig.savefig(figure_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return figure_path


def train_model(model_name: str, model: nn.Module, train_loader: DataLoader, dev_loader: DataLoader, cfg: dict) -> dict:
    checkpoint_path = ARTIFACT_DIR / f"n24news_{model_name.lower()}_best.pt"
    history_path = ARTIFACT_DIR / f"n24news_{model_name.lower()}_history.csv"

    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    total_steps = max(1, len(train_loader) * cfg["epochs"])
    warmup_steps = max(1, int(total_steps * cfg["warmup_ratio"]))
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
    best_macro_f1 = -1.0
    patience_left = cfg["patience"]
    history_rows = []

    for epoch in range(1, cfg["epochs"] + 1):
        model.train()
        train_loss = 0.0
        train_examples = 0

        for batch_inputs, labels in tqdm(train_loader, desc=f"{model_name} epoch {epoch}"):
            batch_inputs, labels = move_batch_to_device(batch_inputs, labels)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(**batch_inputs)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            batch_size = labels.size(0)
            train_loss += loss.item() * batch_size
            train_examples += batch_size

        train_loss /= max(train_examples, 1)
        dev_loss, dev_logits, dev_labels = collect_predictions(model, dev_loader, criterion)
        dev_metrics, _, _ = compute_metrics(dev_logits, dev_labels, label_names)

        history_rows.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "dev_loss": dev_loss,
                "dev_accuracy": dev_metrics["accuracy"],
                "dev_macro_f1": dev_metrics["macro_f1"],
                "dev_weighted_f1": dev_metrics["weighted_f1"],
            }
        )

        print(
            f"{model_name} | epoch {epoch} | train_loss={train_loss:.4f} | "
            f"dev_loss={dev_loss:.4f} | dev_accuracy={dev_metrics['accuracy']:.4f} | "
            f"dev_macro_f1={dev_metrics['macro_f1']:.4f}"
        )

        if dev_metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = dev_metrics["macro_f1"]
            patience_left = cfg["patience"]
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "label_names": label_names,
                    "config": cfg,
                },
                checkpoint_path,
            )
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"Early stopping triggered for {model_name}.")
                break

    history_df = pd.DataFrame(history_rows)
    history_df.to_csv(history_path, index=False)
    history_figure = plot_history(history_df, model_name)

    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    dev_loss, dev_logits, dev_labels = collect_predictions(model, dev_loader, criterion)
    dev_metrics, dev_per_label, dev_cm = compute_metrics(dev_logits, dev_labels, label_names)

    test_loader = train_model.test_loader_lookup[model_name]
    test_loss, test_logits, test_labels = collect_predictions(model, test_loader, criterion)
    test_metrics, test_per_label, test_cm = compute_metrics(test_logits, test_labels, label_names)
    test_metrics["loss"] = float(test_loss)

    confusion_path = plot_confusion(test_cm, model_name, label_names)

    return {
        "model_name": model_name,
        "checkpoint_path": checkpoint_path,
        "history_path": history_path,
        "history_figure": history_figure,
        "history_df": history_df,
        "dev_metrics": dev_metrics,
        "dev_per_label": dev_per_label,
        "dev_confusion": dev_cm,
        "test_metrics": test_metrics,
        "test_per_label": test_per_label,
        "test_confusion": test_cm,
        "confusion_path": confusion_path,
    }


train_model.test_loader_lookup = {}


### Fine-Tune CLIP On The Prepared Split

Cell n?y hu?n luy?n `CLIP` tr?n split c?a `N24News` v? tr? v? metric cu?i tr?n t?p test.


In [ ]:
clip_train_loader = make_loader(train_dataset, CLIP_CFG["batch_size"], clip_collate, shuffle=True)
clip_dev_loader = make_loader(dev_dataset, CLIP_CFG["batch_size"], clip_collate, shuffle=False)
clip_test_loader = make_loader(test_dataset, CLIP_CFG["batch_size"], clip_collate, shuffle=False)
train_model.test_loader_lookup["CLIP"] = clip_test_loader

clip_model = CLIPClassifier(
    backbone=CLIP_CFG["backbone"],
    num_labels=len(label_names),
    dropout=CLIP_CFG["dropout"],
)

clip_result = train_model("CLIP", clip_model, clip_train_loader, clip_dev_loader, CLIP_CFG)
clip_result["test_metrics"]


### Fine-Tune VisualBERT On The Prepared Split

Cell n?y hu?n luy?n `VisualBERT` tr?n c?ng protocol ?? so s?nh c?ng b?ng v?i `CLIP`.


In [ ]:
vb_train_loader = make_loader(train_dataset, VB_CFG["batch_size"], visualbert_collate, shuffle=True)
vb_dev_loader = make_loader(dev_dataset, VB_CFG["batch_size"], visualbert_collate, shuffle=False)
vb_test_loader = make_loader(test_dataset, VB_CFG["batch_size"], visualbert_collate, shuffle=False)
train_model.test_loader_lookup["VisualBERT"] = vb_test_loader

visualbert_model = VisualBERTClassifier(
    visualbert_backbone=VB_CFG["visualbert_backbone"],
    vision_backbone=VB_CFG["vision_backbone"],
    num_labels=len(label_names),
    dropout=VB_CFG["dropout"],
    max_visual_tokens=VB_CFG["max_visual_tokens"],
)

visualbert_result = train_model("VisualBERT", visualbert_model, vb_train_loader, vb_dev_loader, VB_CFG)
visualbert_result["test_metrics"]


### Export The Final Comparison Artifacts

Cell cu?i c?ng t?ng h?p metric, per-class F1 v? c?c file CSV/JSON/PNG ?? ph?c v? b?o c?o.


In [ ]:
comparison_rows = []
for result in [clip_result, visualbert_result]:
    row = {
        "model": result["model_name"],
        **result["test_metrics"],
    }
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows).sort_values(["macro_f1", "accuracy"], ascending=False)
comparison_path = ARTIFACT_DIR / "n24news_model_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

metrics_summary = {
    "dataset": "N24News",
    "task": "image + news text -> category classification",
    "split_sizes": {split_name: int(len(frame)) for split_name, frame in split_frames.items()},
    "num_classes": len(label_names),
    "label_names": label_names,
    "models": {
        "CLIP": clip_result["test_metrics"],
        "VisualBERT": visualbert_result["test_metrics"],
    },
    "sources": {
        "official_repo": OFFICIAL_REPO_URL,
        "paper": PAPER_URL,
        "drive": N24NEWS_DRIVE_URL,
    },
}
metrics_summary_path = ARTIFACT_DIR / "n24news_metrics_summary.json"
metrics_summary_path.write_text(json.dumps(metrics_summary, indent=2, ensure_ascii=False), encoding="utf-8")

per_label_df = clip_result["test_per_label"].merge(
    visualbert_result["test_per_label"][["category", "f1"]],
    on="category",
    suffixes=("_clip", "_visualbert"),
)
per_label_path = ARTIFACT_DIR / "n24news_per_label_f1.csv"
per_label_df.to_csv(per_label_path, index=False)

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(per_label_df))
width = 0.38
ax.bar(x - width / 2, per_label_df["f1_clip"], width=width, label="CLIP", color="#16324f")
ax.bar(x + width / 2, per_label_df["f1_visualbert"], width=width, label="VisualBERT", color="#c59b45")
ax.set_xticks(x)
ax.set_xticklabels(per_label_df["category"], rotation=35, ha="right")
ax.set_ylabel("F1 score")
ax.set_title("Per-class F1 on N24News test split")
ax.legend()
fig.tight_layout()
per_label_figure = ARTIFACT_DIR / "n24news_per_label_f1.png"
fig.savefig(per_label_figure, dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Saved comparison file:", comparison_path)
print("Saved metrics summary:", metrics_summary_path)
print("Saved per-class F1 file:", per_label_path)
print("Saved per-class F1 figure:", per_label_figure)
print("Saved CLIP confusion matrix:", clip_result["confusion_path"])
print("Saved VisualBERT confusion matrix:", visualbert_result["confusion_path"])
comparison_df
